In [49]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [40]:
# Read dataset
import pandas as pd
df = pd.read_csv("D:/ISB/datasets/cleaned_canada.csv")
df.head()

,City,Province,Latitude,Longitude,Price,Bedrooms,Bathrooms,Acreage,Property Type,Square Footage,...,Exterior,Fireplace,Heating,Flooring,Roof,Waterfront,Sewer,Pool,Garden,Balcony
0,Revelstoke,BC,50.976585,-118.173149,839000.0,3.0,2.0,0.00,Condo,891.0,...,NaN,No,heat pump,carpet,NaN,No,municipal,No,No,No
1,Boswell,BC,49.471870,-116.770195,1150000.0,3.0,2.0,0.32,Single Family,1881.0,...,NaN,No,heat pump,NaN,NaN,No,septic,No,No,No
2,West Kelowna,BC,49.825230,-119.603253,149000.0,2.0,1.0,0.00,Single Family,912.0,...,Metal,No,NaN,laminate,tar,No,municipal,No,No,No
3,Kelowna,BC,49.821860,-119.480143,1298000.0,5.0,4.0,0.69,Single Family,4374.0,...,NaN,Yes,forced air,NaN,NaN,No,municipal,No,No,No
4,Maple Ridge,BC,49.221673,-122.596637,759900.0,3.0,2.0,0.00,Condo,1254.0,...,NaN,No,radiant,NaN,NaN,No,none,No,No,No


In [41]:
# Basic overview
print(df.shape)
print(df.info())

(44896, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44896 entries, 0 to 44895
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   City            44896 non-null  object 
 1   Province        44896 non-null  object 
 2   Latitude        44896 non-null  float64
 3   Longitude       44896 non-null  float64
 4   Price           44896 non-null  float64
 5   Bedrooms        44896 non-null  float64
 6   Bathrooms       44896 non-null  float64
 7   Acreage         44896 non-null  float64
 8   Property Type   44896 non-null  object 
 9   Square Footage  44896 non-null  float64
 10  Garage          44896 non-null  object 
 11  Parking         44896 non-null  object 
 12  Basement        14962 non-null  object 
 13  Exterior        17439 non-null  object 
 14  Fireplace       44896 non-null  object 
 15  Heating         38242 non-null  object 
 16  Flooring        15216 non-null  object 
 17  Roof            100

In [42]:
# Select categorical columns (object dtype)

cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Categorical Columns:")
print(cat_cols)




Categorical Columns:
['City', 'Province', 'Property Type', 'Garage', 'Parking', 'Basement', 'Exterior', 'Fireplace', 'Heating', 'Flooring', 'Roof', 'Waterfront', 'Sewer', 'Pool', 'Garden', 'Balcony']


In [43]:
#checking for null values(missing values)
print(df[cat_cols].isnull().sum())

City                 0
Province             0
Property Type        0
Garage               0
Parking              0
Basement         29934
Exterior         27457
Fireplace            0
Heating           6654
Flooring         29680
Roof             34844
Waterfront           0
Sewer                0
Pool                 0
Garden               0
Balcony              0
dtype: int64


In [44]:
# Fill missing categorical values
df[cat_cols] = df[cat_cols].fillna("Unknown")

# Verify missing values removed
print(df[cat_cols].isnull().sum())

City             0
Province         0
Property Type    0
Garage           0
Parking          0
Basement         0
Exterior         0
Fireplace        0
Heating          0
Flooring         0
Roof             0
Waterfront       0
Sewer            0
Pool             0
Garden           0
Balcony          0
dtype: int64


In [45]:
# Check number of unique values per categorical column
for col in cat_cols:
    print(f"{col}: {df[col].nunique()} unique values")

City: 3149 unique values
Province: 11 unique values
Property Type: 9 unique values
Garage: 2 unique values
Parking: 2 unique values
Basement: 4 unique values
Exterior: 7 unique values
Fireplace: 2 unique values
Heating: 10 unique values
Flooring: 8 unique values
Roof: 17 unique values
Waterfront: 2 unique values
Sewer: 5 unique values
Pool: 2 unique values
Garden: 2 unique values
Balcony: 2 unique values


In [46]:
# Binary columns (Yes/No type)
binary_cols = [col for col in cat_cols if df[col].nunique() == 2]
binary_cols

['Garage', 'Parking', 'Fireplace', 'Waterfront', 'Pool', 'Garden', 'Balcony']

In [47]:
# High cardinality columns (like City)
high_card_cols = [col for col in cat_cols if df[col].nunique() > 50]
print("High Cardinal Columns:", high_card_cols)

High Cardinal Columns: ['City']


In [50]:
# Binary Encoding (Yes/No → 1/0)
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'Unknown': 0})

print("Binary Encoding Completed")


Binary Encoding Completed


In [52]:
nominal_cols = [
    col for col in cat_cols
    if col not in binary_cols and col not in high_card_cols
]
# One-Hot Encoding for nominal categorical columns
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print("One-Hot Encoding Completed")

One-Hot Encoding Completed


In [55]:
# Frequency Encoding for high cardinal columns
for col in high_card_cols:
    freq_map = df[col].value_counts(normalize=True)
    df[col + "_freq"] = df[col].map(freq_map)
    
# Drop original high cardinal column
df.drop(columns=high_card_cols, inplace=True)

print("Frequency Encoding Applied to:", high_card_cols)

Frequency Encoding Applied to: ['City']


In [57]:
# Check remaining categorical columns
remaining_cat = df.select_dtypes(include=['object']).columns
print("Remaining Categorical Columns:", remaining_cat)

# Final dataset info
print(df.info())


Remaining Categorical Columns: Index([], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44896 entries, 0 to 44895
Data columns (total 78 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Latitude                         44896 non-null  float64
 1   Longitude                        44896 non-null  float64
 2   Price                            44896 non-null  float64
 3   Bedrooms                         44896 non-null  float64
 4   Bathrooms                        44896 non-null  float64
 5   Acreage                          44896 non-null  float64
 6   Square Footage                   44896 non-null  float64
 7   Garage                           0 non-null      float64
 8   Parking                          0 non-null      float64
 9   Fireplace                        0 non-null      float64
 10  Waterfront                       0 non-null      float64
 11  Pool                   